In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [2]:
event_log_name = "bpic17"
log_path = f"./.out/eventlogs/{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/6801 [00:00<?, ?it/s]

In [3]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [4]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
# if True:
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

File bpic17-0.3-1.decl does not exist, running discovery...
Computing discovery ...
Total constraints discovered: 266


In [5]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


File bpic17_5000_conformance_results.pkl does not exist, running conformance checking...
Conformance checking results saved to bpic17_5000_conformance_results.pkl


In [6]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


In [9]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
# filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.3].sort_values(by=['confidence', "support"], ascending=[False, True])
print(f"Filtered Metrics DataFrame: {event_log_name}")
display(filtered_metrics_df)

Filtered Metrics DataFrame: bpic17


,support,confidence
"Responded Existence[O_Accepted, W_Validate application] | |",0.081753,1.000000
"Responded Existence[O_Accepted, W_Call after offers] | |",0.081753,1.000000
"Responded Existence[O_Accepted, A_Create Application] | |",0.081753,1.000000
"Responded Existence[O_Accepted, W_Complete application] | |",0.081753,1.000000
"Responded Existence[A_Pending, W_Validate application] | |",0.082047,1.000000
...,...,...
"Responded Existence[W_Call after offers, W_Validate application] | |",0.100573,0.017397
"Responded Existence[W_Call after offers, A_Validating] | |",0.100132,0.017320
"Responded Existence[W_Call after offers, O_Returned] | |",0.099103,0.017142
"Responded Existence[W_Call after offers, A_Pending] | |",0.082047,0.014192


In [8]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[O_Accepted, W_Validate application] | |	0.08175268342890751	1.0

In [10]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    # "Responded Existence[Activity BZ, Activity A] | |", # Gigantic
    # "Responded Existence[Activity Q, Activity O] | |", # wide
    "Responded Existence[O_Accepted, W_Validate application] | |", # bpic13
    ]

In [11]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[O_Accepted, W_Validate application] | |    556
dtype: int64
[15, 19, 20, 21, 29, 36, 37, 39, 59, 90, 101, 129, 136, 165, 179, 181, 192, 201, 213, 217, 230, 238, 239, 254, 255, 268, 276, 313, 315, 318, 326, 347, 366, 374, 391, 398, 405, 425, 432, 445, 447, 450, 460, 490, 494, 502, 511, 549, 565, 580, 598, 613, 628, 632, 644, 684, 686, 699, 722, 756, 771, 774, 794, 813, 819, 834, 835, 849, 860, 865, 905, 914, 946, 959, 960, 1046, 1052, 1074, 1075, 1130, 1133, 1167, 1180, 1189, 1192, 1199, 1211, 1233, 1242, 1258, 1295, 1297, 1302, 1314, 1315, 1333, 1345, 1346, 1376, 1378, 1385, 1411, 1414, 1435, 1460, 1486, 1496, 1497, 1511, 1514, 1548, 1563, 1572, 1573, 1576, 1586, 1590, 1593, 1603, 1607, 1643, 1644, 1646, 1654, 1666, 1667, 1668, 1678, 1715, 1756, 1773, 1780, 1794, 1798, 1811, 1825, 1837, 1847, 1881, 1898, 1905, 1922, 1925, 1934, 1938, 1943, 1959, 1980, 1986, 1989, 2000, 2027, 2049, 2056, 2083, 2099, 2111, 2142, 2143, 2149, 2182, 2189, 2192, 2197, 2219, 2222, 2249, 22

In [12]:
print("END")

END


In [ ]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)